# 01 - Multi-Dataset Exploration and Quality Audit

In [ ]:
from pathlib import Path
import json
import os
import sys

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
KAGGLE = Path("/kaggle").exists()
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)
print({"project_root": str(PROJECT_ROOT), "quick_run": QUICK_RUN, "kaggle": KAGGLE})

In [ ]:
from src.data import (
    load_config,
    load_fraud_dataframe,
    make_synthetic_baf_data,
    make_synthetic_fraud_data,
    prepare_dataset,
)

max_rows = 12000 if QUICK_RUN else None
output_dir = OUTPUT_BASE / "01_data_exploration"
output_dir.mkdir(parents=True, exist_ok=True)
dataset_summaries = []

## I. IEEE-CIS

In [ ]:
config = load_config(PROJECT_ROOT / "configs/ieee_cis.yaml")
try:
    frame = load_fraud_dataframe(config, max_rows=max_rows)
    data_source = "real"
except FileNotFoundError:
    if not ALLOW_SYNTHETIC_FALLBACK:
        raise
    frame = make_synthetic_fraud_data(max_rows or 6000, seed=config["project"]["seed"])
    data_source = "synthetic"
print({"dataset": "IEEE-CIS", "data_source": data_source, "rows": len(frame), "columns": frame.shape[1]})
display(frame.head())

In [ ]:
target = config["dataset"]["target_column"]
time_column = config["dataset"]["time_column"]
summary = pd.DataFrame({
    "dataset": ["IEEE-CIS"],
    "rows": [len(frame)],
    "columns": [frame.shape[1]],
    "fraud_count": [int(frame[target].sum())],
    "fraud_rate": [float(frame[target].mean())],
    "duplicate_rows": [int(frame.duplicated().sum())],
    "data_source": [data_source],
})
missing = frame.isna().mean().sort_values(ascending=False).rename("missing_fraction").to_frame()
prepared = prepare_dataset(frame, config)
split_summary = pd.DataFrame([
    {"split": "train", "rows": len(prepared.y_train), "fraud_rate": prepared.y_train.mean(),
     "time_min": prepared.train_frame[time_column].min(), "time_max": prepared.train_frame[time_column].max()},
    {"split": "validation", "rows": len(prepared.y_validation), "fraud_rate": prepared.y_validation.mean(),
     "time_min": prepared.validation_frame[time_column].min(), "time_max": prepared.validation_frame[time_column].max()},
    {"split": "test", "rows": len(prepared.y_test), "fraud_rate": prepared.y_test.mean(),
     "time_min": prepared.test_frame[time_column].min(), "time_max": prepared.test_frame[time_column].max()},
])
dataset_summaries.append(summary.copy())
display(summary, missing.head(20), split_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
split_summary.plot.bar(x="split", y="fraud_rate", ax=axes[0], legend=False, color="#C44E52")
axes[0].set_title("IEEE-CIS fraud rate by split")
axes[0].set_ylabel("Fraud rate")
missing.head(20).sort_values("missing_fraction").plot.barh(ax=axes[1], legend=False, color="#4C72B0")
axes[1].set_title("IEEE-CIS top missing columns")
axes[1].set_xlabel("Missing fraction")
plt.tight_layout()
fig.savefig(output_dir / "ieee_cis_data_quality.png", dpi=160, bbox_inches="tight")
plt.show()

split_summary.to_csv(output_dir / "ieee_cis_split_summary.csv", index=False)
missing.to_csv(output_dir / "ieee_cis_missingness.csv")

## II. BAF

In [ ]:
config = load_config(PROJECT_ROOT / "configs/baf.yaml")
try:
    frame = load_fraud_dataframe(config, max_rows=max_rows)
    data_source = "real"
except FileNotFoundError:
    if not ALLOW_SYNTHETIC_FALLBACK:
        raise
    frame = make_synthetic_baf_data(max_rows or 6000, seed=config["project"]["seed"])
    data_source = "synthetic"
print({"dataset": "BAF", "data_source": data_source, "rows": len(frame), "columns": frame.shape[1]})
display(frame.head())

In [ ]:
target = config["dataset"]["target_column"]
time_column = config["dataset"]["time_column"]
summary = pd.DataFrame({
    "dataset": ["BAF"],
    "rows": [len(frame)],
    "columns": [frame.shape[1]],
    "fraud_count": [int(frame[target].sum())],
    "fraud_rate": [float(frame[target].mean())],
    "duplicate_rows": [int(frame.duplicated().sum())],
    "data_source": [data_source],
})
missing = frame.isna().mean().sort_values(ascending=False).rename("missing_fraction").to_frame()
prepared = prepare_dataset(frame, config)
split_summary = pd.DataFrame([
    {"split": "train", "rows": len(prepared.y_train), "fraud_rate": prepared.y_train.mean(),
     "time_min": prepared.train_frame[time_column].min(), "time_max": prepared.train_frame[time_column].max()},
    {"split": "validation", "rows": len(prepared.y_validation), "fraud_rate": prepared.y_validation.mean(),
     "time_min": prepared.validation_frame[time_column].min(), "time_max": prepared.validation_frame[time_column].max()},
    {"split": "test", "rows": len(prepared.y_test), "fraud_rate": prepared.y_test.mean(),
     "time_min": prepared.test_frame[time_column].min(), "time_max": prepared.test_frame[time_column].max()},
])
dataset_summaries.append(summary.copy())
display(summary, missing.head(20), split_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
split_summary.plot.bar(x="split", y="fraud_rate", ax=axes[0], legend=False, color="#C44E52")
axes[0].set_title("BAF fraud rate by split")
axes[0].set_ylabel("Fraud rate")
missing.head(20).sort_values("missing_fraction").plot.barh(ax=axes[1], legend=False, color="#4C72B0")
axes[1].set_title("BAF top missing columns")
axes[1].set_xlabel("Missing fraction")
plt.tight_layout()
fig.savefig(output_dir / "baf_data_quality.png", dpi=160, bbox_inches="tight")
plt.show()

split_summary.to_csv(output_dir / "baf_split_summary.csv", index=False)
missing.to_csv(output_dir / "baf_missingness.csv")

## III. So sánh tổng quan

In [ ]:
combined_summary = pd.concat(dataset_summaries, ignore_index=True)
display(combined_summary)
ax = combined_summary.plot.bar(x="dataset", y="fraud_rate", legend=False, color=["#4C72B0", "#55A868"])
ax.set_title("Fraud rate across datasets")
ax.set_ylabel("Fraud rate")
plt.tight_layout()
plt.savefig(output_dir / "dataset_fraud_rate_comparison.png", dpi=160, bbox_inches="tight")
plt.show()
combined_summary.to_csv(output_dir / "dataset_summary.csv", index=False)